In [1]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

C:\Users\USER\AppData\Local\Temp\ipykernel_19660\3933654057.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
d:\Projects\rag\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# read all the pdf's inside the folder
def process_all_pdfs(pdf_directory):
    all_documents = []
    pdf_dir = Path(pdf_directory)

    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    print(f"Found {len(pdf_files)} PDF files to process")

    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()

            # add source information to metadata
            for doc in documents:
                doc.metadata['source-file'] = pdf_file.name
                doc.metadata['file-type'] = 'pdf'

            all_documents.extend(documents)
            print(f"loaded {len(documents)} pages")

        except Exception as e:
            print(f"error {e}")

    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

all_pdf_documents = process_all_pdfs("../data")


Found 8 PDF files to process

Processing: 01. Bidirection-RNN.pdf
loaded 2 pages

Processing: 01. EncoderDecoderSeq2SEq.pdf
loaded 4 pages

Processing: Correlation Analysis.pdf
loaded 9 pages

Processing: Exponential distribution.pdf
loaded 6 pages

Processing: Poisson Distribution.pdf
loaded 11 pages

Processing: Regression Analysis.pdf
loaded 4 pages

Processing: Test of Hypothesis.pdf
loaded 11 pages

Processing: Theory of Probability.pdf
loaded 12 pages

Total documents loaded: 59


In [3]:
all_pdf_documents

[Document(metadata={'producer': 'PyPDF', 'creator': 'PyPDF', 'creationdate': '', 'source': '..\\data\\pdf\\01. Bidirection-RNN.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1', 'source-file': '01. Bidirection-RNN.pdf', 'file-type': 'pdf'}, page_content=''),
 Document(metadata={'producer': 'PyPDF', 'creator': 'PyPDF', 'creationdate': '', 'source': '..\\data\\pdf\\01. Bidirection-RNN.pdf', 'total_pages': 2, 'page': 1, 'page_label': '2', 'source-file': '01. Bidirection-RNN.pdf', 'file-type': 'pdf'}, page_content=''),
 Document(metadata={'producer': 'PyPDF', 'creator': 'PyPDF', 'creationdate': '', 'source': '..\\data\\pdf\\01. EncoderDecoderSeq2SEq.pdf', 'total_pages': 4, 'page': 0, 'page_label': '1', 'source-file': '01. EncoderDecoderSeq2SEq.pdf', 'file-type': 'pdf'}, page_content=''),
 Document(metadata={'producer': 'PyPDF', 'creator': 'PyPDF', 'creationdate': '', 'source': '..\\data\\pdf\\01. EncoderDecoderSeq2SEq.pdf', 'total_pages': 4, 'page': 1, 'page_label': '2', 'source-file': 

In [10]:
# Text splitting get into chunks
def split_documents(documents, chunk_size=1000, chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size = chunk_size,
        chunk_overlap = chunk_overlap,
        length_function = len,
        separators=["\n\n", "\n", " ", ""]
    )

    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")

    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")

    return split_docs

In [11]:
chunks = split_documents(all_pdf_documents)
chunks

Split 59 documents into 80 chunks

Example chunk:
Content: STT211: Engineering Statistics & Complex Variable Lectured by
Md. Kaderi Kibria, STT-HSTU
Chapter # 4           Correlation Analysis
Introduction
So  far  we  have  confined  our  discussion  to  univ...
Metadata: {'producer': 'LibreOffice 7.1', 'creator': 'Writer', 'creationdate': '2024-02-01T01:23:25+06:00', 'source': '..\\data\\pdf\\Correlation Analysis.pdf', 'total_pages': 9, 'page': 0, 'page_label': '1', 'source-file': 'Correlation Analysis.pdf', 'file-type': 'pdf'}


[Document(metadata={'producer': 'LibreOffice 7.1', 'creator': 'Writer', 'creationdate': '2024-02-01T01:23:25+06:00', 'source': '..\\data\\pdf\\Correlation Analysis.pdf', 'total_pages': 9, 'page': 0, 'page_label': '1', 'source-file': 'Correlation Analysis.pdf', 'file-type': 'pdf'}, page_content='STT211: Engineering Statistics & Complex Variable Lectured by\nMd. Kaderi Kibria, STT-HSTU\nChapter # 4           Correlation Analysis\nIntroduction\nSo  far  we  have  confined  our  discussion  to  univariate  distributions  only  i.e.,  the  distributions\ninvolving only one variable and also saw how the various measures of central tendency, dispersion,\nskewness and kurtosis can be used for the purposes of comparison and analysis. We may, however,\ncome across certain series where each item of the series may assume the values of two or more\nvariables. If we measure the heights and weights of n individuals, we obtain a series in which each\nunit (individual) of the series assumes two values—